<div class="alert alert-block alert-success">
<h1> Machine Learning - Project Cars4You</h1>
<h2> 3 - Modeling<h2>
<h2> Group 36 </h2>
<h3> 20225/2026 </h3>

***

## **Table of Contents**<br>

[1. **Importing libraries & Data**](#1-Importing-libraries-&-Data)<br>

[2. Modeling](#2-Modeling)<br>
- [R**2](#R**")<br>
- [MAE](#MAE")<br>
- [MSE](#MSE")<br>
- [RMSE](#RMSE")<br>
- [MedAE](#MedAE")<br>
- [MAPE](#MAPE")<br>
[2.1 Comparing Diferences](#2.1-Comparing-Diferences")<br>

[3. Final Predictions](#3-Final-Predictions)<br>
    

***

This notebook will consist of Modeling and making final predictions.

> Student Name - Artem Polikarpov
>> 20250443

> Student Name - Diogo Montenegro
>> 20250491

> Student Name - Francisco Martins
>> 20250482

> Student Name - João Cardoso
>> 20240529 


***

# 1. Import Libraries & Data

In [2]:
# general imports that we will need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
sns.set()

from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

#grid
from sklearn.model_selection import ParameterGrid
#
from sklearn.model_selection import ParameterSampler
# ignore warnings
import warnings
warnings.filterwarnings('ignore')

#functions
from models_func import ( get_results, inverse_y_log1p, random_search, results_to_df, make_submission, predict_results)


#set random seed for reproducibility
RSEED = 42
np.random.seed(RSEED)
pd.set_option('display.max_rows', 500)


Before starting, for every model that we performed a hyperparameter tunning, first we created a grid that covers a wide and meaningful range of configuration options. This grid allows a systematic search over different model complexities, tree shapes and sampling strategies.

Then, to efficiently explore the hyperparameter space defined for the models, we implemented a **random search strategy**. Instead of evaluating every possible combination in the full grid, this method samples a fixed number of configurations at random, providing a good trade off between computational cost and search breadth. If the parameters of the chosen models are not shown in the random search output, it's because at some point we did run that cell again and it gave different models.

We did not use the `"RandomizedSearchCV"`(https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) or the `"GridSearchcv"`(https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) because, since we are using the target with the log transformation, using this functions would give us MAE scores not interpretable in the context of our project. 

#### Data Loading

First, we imported the data from the "Holdout" notebook.

In [3]:
X_train_filtered = pd.read_csv('../project_data/X_train_filtered_data_KNN.csv', index_col = 0)
y_train = pd.read_csv('../project_data/y_train_data_KNN.csv', index_col=0)

X_val_filtered = pd.read_csv('../project_data/X_val_filtered_data_KNN.csv', index_col=0)
y_val = pd.read_csv('../project_data/y_val_data_KNN.csv', index_col=0)

test_filtered = pd.read_csv('../project_data/test_filtered_data_KNN.csv', index_col=0)

Then, we moved all the features selected into a separated list.

In [4]:
cols_selected = X_train_filtered.columns.to_list()
cols_selected

['model_te',
 'engineSize',
 'Brand_opel',
 'mpg',
 'transmission_manual',
 'mileage',
 'Brand_audi',
 'fuelType_petrol',
 'carAge',
 'Brand_volkswagen',
 'Brand_mercedes-benz',
 'Brand_hyundai',
 'Brand_bmw',
 'tax_bin_101_170',
 'tax_bin_171_450',
 'tax_bin_1_100',
 'transmission_semi-auto',
 'transmission_automatic',
 'fuelType_diesel',
 'fuelType_hybrid']

# 2. Modeling

Before starting training any models, we decided to make the **reverse log transformation** so we could interpret better the results gave by the model. The model is still training for the target on log, but then we predict on the original target to extract more useful and realistic information.

In [5]:
y_train_orig = np.expm1(y_train)
y_val_orig = np.expm1(y_val)

Before starting training models, the metric we used to evaluate our models was, everytime, the `MAE (Mean Absolute Error)` since, in the context of our project, is the one that gives us the most useful and realistic information. We also included the R squared score just if there is any useful insights that this score coud give us.

## Linear Regression 

The first model we trained was actually one of the most basic ones, the `Linear Regression model`.

In [6]:
LR = LinearRegression().fit(X_train_filtered, y_train)

LR_results = predict_results(model = LR, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
LR_results

,Metric,Train,Validation
0,MAE,2198.073,2210.504
1,R^2,0.763,0.746
2,Adj R^2,0.762,0.746


As we can see, the model had a very similar performance both on train and on validation datasets, only with a slight overfit. However, we can clearly see that this is not a good model for our problem, since the MAE we got was 2210.069 on validation. This is, of course, not a godd result since, in the real world, when predicting car prices, we would be missing the real price, on average, by 2210.069£, which is quite a lot.

## Lasso

The next model evaluated was the `Lasso` model, an L1-regularised linear method that can reduce overfitting by shrinking some coefficients to zero.

In [7]:
LassoR = Lasso(alpha =0.001).fit(X_train_filtered, y_train)

LR_results = predict_results(model = LassoR, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
LR_results

,Metric,Train,Validation
0,MAE,2227.077,2244.383
1,R^2,0.707,0.680
2,Adj R^2,0.707,0.680


As observed above, the results for the Lasso model is very similar to the Linear Regression model, either on results or overfitting. Again, having 2244.383£ as MAE, is far from being good on predicting car prices.

## KNeighborsRegressor

The next model evaluated was the `K Nearest Neighbors Regressor`, which estimates the target value by looking at the most similar observations in the training set.

In [8]:
KNN=KNeighborsRegressor(n_neighbors=8, 
                    weights='uniform', 
                    algorithm='auto',  
                    p=2, 
                    metric='minkowski', 
                    metric_params=None, 
                    n_jobs=None).fit(X_train_filtered, y_train)

KNN_results = predict_results(model = KNN, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
KNN_results

KeyboardInterrupt: 

This model gave us completely different results when compared to the previous ones. As we can see, the validation score improved a lot, around 1000£, however, the overfitting also increased, almost 200£ difference between validation and train scores. In a business prespective, 1457.943£ is still not ideal, but it is way better than the previous scores gotten. As expected, because all the limitations that the knn has, such as its inability to extrapolate, this will not be the best model for our project.

## Random Forest Regressor

The next model trained is the `Random Forest Regressor`, a bagging based ensemble method composed of multiple decision trees.

First, as told before, we defined the parameters grid and then we ran this grid with a random search to analyze the different models obtained.

In [ ]:
RF_param_grid = {
    "n_estimators": [800, 1000],          
    "max_depth": [18, 21, 25],         
    "min_samples_split": [7,10],        
    "min_samples_leaf": [1],
    "max_leaf_nodes": [None, 4500, 5000], 
    "min_impurity_decrease": [0.0],      
    "max_features": [0.5, 0.6],       
    "bootstrap": [True],
    "oob_score":[True, False],         
}

RF_grid = list(ParameterGrid(RF_param_grid))

print(f"Nº de combinações: {len(RF_grid)}")

Nº de combinações: 144


In [ ]:
RF_grid_results = random_search(
    model_class=RandomForestRegressor,
    grid=RF_grid,
    X_train=X_train_filtered,
    y_train=y_train,          # log scale
    X_val=X_val_filtered,
    y_val=y_val,              # log scale
    y_train_orig=y_train_orig,  # $ scale
    y_val_orig=y_val_orig,      # $ scale
    base_params={"random_state": 42, "n_jobs": -1},
    inverse_y_func=inverse_y_log1p,
    n_iter=20,
    random_state = None,
    verbose=False,
)

results_to_df(RF_grid_results, sort_by="mae_val", ascending=True, metrics_first=True)


--> Train model 1/20 with params (idx=121): {'bootstrap': True, 'max_depth': 25, 'max_features': 0.6, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 7, 'n_estimators': 800, 'oob_score': False}

--> Train model 2/20 with params (idx=38): {'bootstrap': True, 'max_depth': 18, 'max_features': 0.6, 'max_leaf_nodes': 4500, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 1000, 'oob_score': True}

--> Train model 3/20 with params (idx=10): {'bootstrap': True, 'max_depth': 18, 'max_features': 0.5, 'max_leaf_nodes': 4500, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 7, 'n_estimators': 1000, 'oob_score': True}

--> Train model 4/20 with params (idx=90): {'bootstrap': True, 'max_depth': 21, 'max_features': 0.6, 'max_leaf_nodes': 5000, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 7, 'n_estimators': 1000, 'oob_score': True}

--> Train model 5/20 

,mae_train,mae_val,gap,bootstrap,max_depth,max_features,max_leaf_nodes,min_impurity_decrease,min_samples_leaf,min_samples_split,n_estimators,oob_score,iter
0,766.049,1222.091,456.042,True,25,0.6,NaN,0.0,1,7,800,False,1
1,869.805,1228.098,358.293,True,18,0.6,NaN,0.0,1,7,800,True,10
2,884.472,1228.601,344.129,True,18,0.5,NaN,0.0,1,7,1000,False,18
3,884.852,1229.100,344.248,True,18,0.5,NaN,0.0,1,7,800,True,11
4,919.809,1229.831,310.022,True,21,0.6,5000.0,0.0,1,7,1000,True,4
5,919.869,1229.875,310.006,True,21,0.6,5000.0,0.0,1,7,800,False,7
6,874.896,1229.948,355.052,True,21,0.6,NaN,0.0,1,10,1000,False,17
7,940.165,1234.019,293.854,True,25,0.6,4500.0,0.0,1,7,1000,True,13
8,933.902,1234.564,300.662,True,21,0.5,5000.0,0.0,1,7,1000,False,14
9,942.814,1234.593,291.779,True,21,0.6,4500.0,0.0,1,7,1000,False,16


After analyzing the differents models obtained in the previous cell, we decided to chose one that balances the best either a good score and a "good" overfitting. 

In [ ]:
RF = RandomForestRegressor(
            n_estimators = 1000,
            max_depth = 25,
            min_samples_leaf = 1,
            min_samples_split = 10,        
            max_features = 0.5,
            max_leaf_nodes = 4500,    
            bootstrap = True,
            oob_score = True,
            random_state = 42,
            n_jobs = -1
).fit(X_train_filtered, y_train)

RF_results = predict_results(model = RF, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
RF_results

,Metric,Train,Validation
0,MAE,963.498,1240.449
1,R^2,0.970,0.953
2,Adj R^2,0.970,0.953


As observed, we managed to improve our MAE score a lot, when compared to the previous best model score. Even tough, random forest is an ensemble of many decision trees and this ones itselves are very willing to overfit, the overfit got with this model is still to high so we must search for other type of ensembling, wihtout being bagging.

## HistGradientBoostRegressor

After experimenting with bagging ensemble models, we then evaluated some `boosting` based models. The main difference between these two approaches is that bagging trains multiple models in parallel, while boosting trains models sequentially, where each new model attempts to correct the errors made by the previous one.

The first `boosting`model we used was `HistGradientBoostRegressor`. As previously mentioned, we first defined a parameters grid and then ran it with a random search to help us choosing the best model.

In [ ]:
HGB_param_dist = {
    # Learning rate: how fast the model learns (smaller = slower but often better)
    "learning_rate": [0.05, 0.06, 0.07],
    # Maximum depth of each individual tree (None = unlimited)
    "max_depth": [None, 14, 12],
    # Number of boosting iterations (trees)
    "max_iter": [900, 1000, 1100, 1200],
    # Controls the number of leaves; more leaves = more complex trees
    "max_leaf_nodes": [40, 50, 70, 80],
    # Minimum number of samples per leaf; higher values = more regularization
    "min_samples_leaf": [10, 13, 15, 17, 20],
    # L2 regularization strength on the leaf values
    "l2_regularization": [0, 0.1],
    # Number of bins used for histogram-based splits
    "max_bins": [255],
    "validation_fraction" : [0.1],
    "early_stopping" : [False],
    "n_iter_no_change" : [10, 15]  
}

HGB_grid = list(ParameterGrid(HGB_param_dist))

print(f"Nº de combinações: {len(HGB_grid)}")

Nº de combinações: 2880


In [ ]:
HGBoost_grid_results = random_search(
    model_class=HistGradientBoostingRegressor,
    grid=HGB_grid,
    X_train=X_train_filtered,
    y_train=y_train,          # log scale
    X_val=X_val_filtered,
    y_val=y_val,              # log scale
    y_train_orig=y_train_orig,  # $ scale
    y_val_orig=y_val_orig,      # $ scale
    base_params={"random_state": 42, "loss": "squared_error"},
    inverse_y_func=inverse_y_log1p,
    n_iter=30,
    random_state = None,
    verbose=False,
)

results_to_df(HGBoost_grid_results, sort_by="mae_val", ascending=True, metrics_first=True)



--> Train model 1/30 with params (idx=2244): {'early_stopping': False, 'l2_regularization': 0.1, 'learning_rate': 0.06, 'max_bins': 255, 'max_depth': 12, 'max_iter': 900, 'max_leaf_nodes': 40, 'min_samples_leaf': 15, 'n_iter_no_change': 10, 'validation_fraction': 0.1}

--> Train model 2/30 with params (idx=755): {'early_stopping': False, 'l2_regularization': 0, 'learning_rate': 0.06, 'max_bins': 255, 'max_depth': 14, 'max_iter': 1100, 'max_leaf_nodes': 80, 'min_samples_leaf': 15, 'n_iter_no_change': 15, 'validation_fraction': 0.1}

--> Train model 3/30 with params (idx=2393): {'early_stopping': False, 'l2_regularization': 0.1, 'learning_rate': 0.06, 'max_bins': 255, 'max_depth': 12, 'max_iter': 1200, 'max_leaf_nodes': 80, 'min_samples_leaf': 13, 'n_iter_no_change': 15, 'validation_fraction': 0.1}

--> Train model 4/30 with params (idx=1475): {'early_stopping': False, 'l2_regularization': 0.1, 'learning_rate': 0.05, 'max_bins': 255, 'max_depth': None, 'max_iter': 900, 'max_leaf_nodes':

,mae_train,mae_val,gap,early_stopping,l2_regularization,learning_rate,max_bins,max_depth,max_iter,max_leaf_nodes,min_samples_leaf,n_iter_no_change,validation_fraction,iter
0,933.006,1190.256,257.250,False,0.0,0.06,255,14.0,1100,80,15,15,0.1,2
1,911.914,1193.200,281.286,False,0.0,0.07,255,12.0,1100,70,10,15,0.1,9
2,965.669,1195.866,230.197,False,0.1,0.06,255,NaN,900,80,17,15,0.1,17
3,916.693,1196.280,279.587,False,0.1,0.06,255,12.0,1200,80,13,15,0.1,3
4,897.693,1196.629,298.936,False,0.1,0.07,255,12.0,1200,80,15,10,0.1,30
5,943.538,1196.956,253.418,False,0.0,0.06,255,14.0,1000,80,13,15,0.1,7
6,913.818,1197.801,283.983,False,0.1,0.06,255,14.0,1100,80,10,15,0.1,6
7,968.503,1199.658,231.155,False,0.0,0.06,255,14.0,1100,70,20,15,0.1,19
8,981.952,1200.741,218.789,False,0.0,0.06,255,14.0,900,70,10,10,0.1,27
9,1003.257,1200.926,197.669,False,0.0,0.07,255,NaN,1000,50,15,15,0.1,22


This time, we did not only choose 1 model, but 3 different HistGradientBoostRegressor models. We decided this way, not only to test them individually but also, in the future, to test how they perform while stacked. We decided to choose one that has a very good overall score even if it does overfit more than supposed. One that is well balacend between MAE values and overfitting. Finnally, the last, we chose one that still had a nice score on validation MAE but had much less overfitting. So, we came up with this 3 models and scores :

In [ ]:
HGBoost2 = HistGradientBoostingRegressor(
            learning_rate = 0.06,
            max_iter = 900,
            #max_depth = 12,
            min_samples_leaf = 17,
            max_leaf_nodes = 80,
            max_bins = 255,
            l2_regularization = 0.1,       
            early_stopping = False,    
            validation_fraction = 0.1,
            n_iter_no_change = 15,
            random_state = 42
            ).fit(X_train_filtered, y_train)

HGBoost_results = predict_results(model = HGBoost2, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
HGBoost_results

,Metric,Train,Validation
0,MAE,965.669,1195.866
1,R^2,0.973,0.959
2,Adj R^2,0.973,0.959


In [ ]:
HGBoost6 = HistGradientBoostingRegressor(
            learning_rate = 0.07,
            max_iter = 1100,
            #max_depth = 12,
            min_samples_leaf = 17,
            max_leaf_nodes = 50,
            max_bins = 255,
            l2_regularization = 0.0,       
            early_stopping = False,    
            validation_fraction = 0.1,
            n_iter_no_change = 15,
            random_state = 42
            ).fit(X_train_filtered, y_train)

HGBoost_results = predict_results(model = HGBoost6, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
HGBoost_results

,Metric,Train,Validation
0,MAE,988.656,1197.618
1,R^2,0.972,0.960
2,Adj R^2,0.972,0.960


In [ ]:
HGBoost14 = HistGradientBoostingRegressor(
            learning_rate = 0.05,
            max_iter = 1000,
            #max_depth = 14,
            min_samples_leaf = 17,
            max_leaf_nodes = 70,
            max_bins = 255,
            l2_regularization = 0.0,       
            early_stopping = False,    
            validation_fraction = 0.1,
            n_iter_no_change = 15,
            random_state = 42
            ).fit(X_train_filtered, y_train)

HGBoost_results = predict_results(model = HGBoost14, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
HGBoost_results

,Metric,Train,Validation
0,MAE,1005.366,1204.143
1,R^2,0.971,0.959
2,Adj R^2,0.971,0.959


As we can see, those are our best models so far. When comparing the MAE, they are the ones who gave us thee best results but also on overfitting, it does still overfit, but in a more controled way. Having around 1190-1205£ MAE, in a business prespective, it is already a quite "good" average error for predicting a car price. This model delivered the best results, as expected, because histogram based gradient boosting trains trees sequentially, with each one correcting the errors made by the previous ones. This iterative error-refinement allows the model to capture nonlinear relationships and complex feature interactions that simpler methods cannot learn effectively.

## GradientBoostingRegressor

Then we tested one more type of ensembling boosting, the `GradientBoostingRegressor` which is smiliar to the previous model but the main difference is how they handle continuous features. While this method evaluates all possible split points, the histogram based version groups values into bins, making the training process significantly faster and more scalable.

We proceeded the same way as before, defining first the parameters grid and then running a random search.

In [ ]:
GBoost_param_grid = {
    "learning_rate" : [0.08, 0.1],
    "n_estimators" : [300, 400, 500],
    "max_depth" : [8, 10],
    "min_samples_leaf" : [10, 15],
    "min_samples_split" : [10, 15, 20],
    "max_features" : [0.5, 0.6, 0.8],
    "subsample" : [0.8]        
}

GBoost_grid = list(ParameterGrid(GBoost_param_grid))

print(f"Nº de combinações: {len(GBoost_grid)}")

Nº de combinações: 216


In [ ]:
GBoost_grid_results = random_search(
    model_class=GradientBoostingRegressor,
    grid=GBoost_grid,
    X_train=X_train_filtered,
    y_train=y_train,          # log scale
    X_val=X_val_filtered,
    y_val=y_val,              # log scale
    y_train_orig=y_train_orig,  # $ scale
    y_val_orig=y_val_orig,      # $ scale
    base_params={"random_state": 42, "loss": "squared_error"},
    inverse_y_func=inverse_y_log1p,
    n_iter=15,
    random_state = None,
    verbose=False,
)

results_to_df(GBoost_grid_results, sort_by="mae_val", ascending=True, metrics_first = True)


--> Train model 1/15 with params (idx=107): {'learning_rate': 0.08, 'max_depth': 10, 'max_features': 0.8, 'min_samples_leaf': 15, 'min_samples_split': 20, 'n_estimators': 500, 'subsample': 0.8}

--> Train model 2/15 with params (idx=148): {'learning_rate': 0.1, 'max_depth': 8, 'max_features': 0.8, 'min_samples_leaf': 10, 'min_samples_split': 15, 'n_estimators': 400, 'subsample': 0.8}

--> Train model 3/15 with params (idx=20): {'learning_rate': 0.08, 'max_depth': 8, 'max_features': 0.6, 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 500, 'subsample': 0.8}

--> Train model 4/15 with params (idx=93): {'learning_rate': 0.08, 'max_depth': 10, 'max_features': 0.8, 'min_samples_leaf': 10, 'min_samples_split': 15, 'n_estimators': 300, 'subsample': 0.8}

--> Train model 5/15 with params (idx=71): {'learning_rate': 0.08, 'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 15, 'min_samples_split': 20, 'n_estimators': 500, 'subsample': 0.8}

--> Train model 6/15 with para

,mae_train,mae_val,gap,learning_rate,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,subsample,iter
0,863.437,1192.913,329.476,0.10,10.0,0.5,10.0,10.0,500.0,0.8,14
1,850.617,1193.557,342.940,0.10,10.0,0.6,10.0,20.0,500.0,0.8,9
2,900.581,1197.306,296.725,0.10,10.0,0.6,10.0,10.0,400.0,0.8,11
3,975.193,1204.309,229.116,0.08,10.0,0.5,15.0,20.0,500.0,0.8,5
4,939.600,1205.455,265.855,0.08,10.0,0.8,10.0,10.0,400.0,0.8,15
5,970.629,1210.258,239.629,0.08,10.0,0.8,15.0,20.0,500.0,0.8,1
6,983.089,1214.333,231.244,0.10,10.0,0.5,15.0,15.0,400.0,0.8,10
7,975.302,1214.518,239.216,0.10,10.0,0.8,15.0,15.0,400.0,0.8,6
8,999.315,1214.885,215.570,0.08,10.0,0.6,10.0,15.0,300.0,0.8,8
9,991.699,1215.166,223.467,0.08,10.0,0.8,10.0,15.0,300.0,0.8,4


As we can observe, we were also able to achieve very good results but, unfortunately, with much more overfitting when comparing to the **HistGradientBoostRegressor**. We tried not only choosing the parameters that gave us the best score, but also balancing this with a controlled overfitting. We came up with the following model : 

In [ ]:
GBoost10 = GradientBoostingRegressor(
            learning_rate = 0.1,
            n_estimators = 300,
            min_samples_leaf = 15,
            min_samples_split = 20,        
            max_depth = 10,    
            max_features = 0.6,
            subsample = 0.8,
            random_state = 42
            ).fit(X_train_filtered, y_train)

GBoost_results = predict_results(model = GBoost10, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig)
GBoost_results

,Metric,Train,Validation
0,MAE,1016.066,1217.943
1,R^2,0.971,0.959
2,Adj R^2,0.971,0.959


Even not being the best model so far, it also gave very good results in the context of our project. As said before, it was expected giving very good results when compared, for example, to linear regression or even random fores regressor, but it was also expected not giving better results when compared to the hist gradient boost regressor.

## MLP Regressor

For the last single model, we tried to implement a `neural network based approach using an MLPRegressor`. Unlike tree based ensemble methods, a multilayer perceptron attempts to learn complex nonlinear relationships directly through multiple layers of interconnected neurons. To give this model a fair chance, we designed a relatively broad hyperparameter search, exploring different hidden layer architectures, learning rates, regularisation strengths, and training settings. The idea was to assess whether a neural network could capture patterns that tree models might overlook.

In [ ]:
MLP_param_grid = {"hidden_layer_sizes":[
        (128,128,128,128),
        (100,100,100,100),
        (8,16,32,64,128)],
    "alpha":[0.0001,0.0005,0.001],
    "learning_rate_init" : [0.005, 0.01,0.001],
    "n_iter_no_change": [20,15,10],
    "validation_fraction": [0.2,0.1,0.0],
    "max_iter":[1000,1100,900],       
}

MLP_grid = list(ParameterGrid(MLP_param_grid))

print(f"Nº de combinações: {len(MLP_grid)}")

Nº de combinações: 729


In [ ]:
MLP_grid_results = random_search(
    model_class=MLPRegressor,
    grid=MLP_grid,
    X_train=X_train_filtered,
    y_train=y_train,          # log scale
    X_val=X_val_filtered,
    y_val=y_val,              # log scale
    y_train_orig=y_train_orig,  # $ scale
    y_val_orig=y_val_orig,      # $ scale
    base_params={"random_state": 42,
                 "activation": 'relu',        
                 "solver": 'adam',
                 "learning_rate": 'adaptive'},
    inverse_y_func=inverse_y_log1p,
    n_iter=5,
    random_state = None,
    verbose=False,
)

results_to_df(MLP_grid_results, sort_by="mae_val", ascending=True, metrics_first=True)


--> Train model 1/5 with params (idx=715): {'alpha': 0.001, 'hidden_layer_sizes': (8, 16, 32, 64, 128), 'learning_rate_init': 0.001, 'max_iter': 1100, 'n_iter_no_change': 15, 'validation_fraction': 0.1}

--> Train model 2/5 with params (idx=17): {'alpha': 0.0001, 'hidden_layer_sizes': (128, 128, 128, 128), 'learning_rate_init': 0.005, 'max_iter': 1100, 'n_iter_no_change': 10, 'validation_fraction': 0.0}

--> Train model 3/5 with params (idx=112): {'alpha': 0.0001, 'hidden_layer_sizes': (100, 100, 100, 100), 'learning_rate_init': 0.01, 'max_iter': 1000, 'n_iter_no_change': 15, 'validation_fraction': 0.1}

--> Train model 4/5 with params (idx=591): {'alpha': 0.001, 'hidden_layer_sizes': (100, 100, 100, 100), 'learning_rate_init': 0.005, 'max_iter': 900, 'n_iter_no_change': 10, 'validation_fraction': 0.2}

--> Train model 5/5 with params (idx=341): {'alpha': 0.0005, 'hidden_layer_sizes': (100, 100, 100, 100), 'learning_rate_init': 0.005, 'max_iter': 1100, 'n_iter_no_change': 10, 'validat

,mae_train,mae_val,gap,alpha,hidden_layer_sizes,learning_rate_init,max_iter,n_iter_no_change,validation_fraction,iter
0,1475.542,1524.779,49.237,0.0001,"(100, 100, 100, 100)",0.010,1000,15,0.1,3
1,2011.319,2025.227,13.908,0.0010,"(8, 16, 32, 64, 128)",0.001,1100,15,0.1,1
2,2038.655,2093.593,54.938,0.0005,"(100, 100, 100, 100)",0.005,1100,10,0.0,5
3,2266.756,2323.865,57.109,0.0010,"(100, 100, 100, 100)",0.005,900,10,0.2,4
4,2809.473,2829.226,19.753,0.0001,"(128, 128, 128, 128)",0.005,1100,10,0.0,2


After running the random search, we decided to como up with the following model :

In [ ]:
MLP = MLPRegressor(
            hidden_layer_sizes = (128,128,128,128),
            activation = 'relu',
            alpha = 0.001,       
            solver = 'adam',    
            learning_rate = 'adaptive',
            n_iter_no_change = 20,        
            validation_fraction = 0.0,    
            max_iter = 1100,
            learning_rate_init = 0.005,
            random_state = 42
            ).fit(X_train_filtered, y_train)

MLP_results = predict_results(model = MLP, X_train = X_train_filtered, X_val = X_val_filtered,  y_train_orig = y_train_orig, y_val_orig = y_val_orig, MLP = False)
MLP_results

,Metric,Train,Validation
0,MAE,1439.030,1494.476
1,R^2,0.935,0.932
2,Adj R^2,0.935,0.932


As expected, the results were not the best but it did overfit less than the tree based models. Even not being the best model, it still gives us a better average error for predicting car prices.

## StackingRegressor

Finnally, we can move on for our last model, which is acctually a stacking of different models trained above. This model is epected to achieve the best results because the `Stacking Regressor` is an ensemble learning technique that combines the strengths of multiple different models. Instead of relying on a single algorithm, stacking trains several base models in parallel and then uses a final model, called the meta learner, to combine their predictions. The key advantage of stacking is that it allows the ensemble to exploit complementary strengths of multiple algorithms.

We chose every model trained before as the base estimators and then, we kept removing/adding each model each iteration until achieving the best result. The best model is based on `HGB2`, `HGB6`and `GBoost10`.

In [ ]:
# 2) Modelos base
base_estimators = [

    ("hgb", HistGradientBoostingRegressor(
            learning_rate = 0.06,
            max_iter = 900,
            #max_depth = 12,
            min_samples_leaf = 17,
            max_leaf_nodes = 80,
            max_bins = 255,
            l2_regularization = 0.1,       
            early_stopping = False,    
            validation_fraction = 0.1,
            n_iter_no_change = 15,
            random_state = 42
    )),
    ("gb", GradientBoostingRegressor(
            learning_rate = 0.1,
            n_estimators = 300,
            min_samples_leaf = 15,
            min_samples_split = 20,        
            max_depth = 10,    
            max_features = 0.6,
            subsample = 0.8,
            random_state = 42
    )),

    ("hgb2",HistGradientBoostingRegressor(
            learning_rate = 0.07,
            max_iter = 1100,
            #max_depth = 12,
            min_samples_leaf = 17,
            max_leaf_nodes = 50,
            max_bins = 255,
            l2_regularization = 0.0,       
            early_stopping = False,    
            validation_fraction = 0.1,
            n_iter_no_change = 15,
            random_state = 42
            )),
 
]

# 3) Meta-modelo (camada 2) – simples e rápido
meta_model = LinearRegression()

# 4) Stacking Regressor
stack = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_model,
    passthrough=False,  # se True, o meta-model também recebe as features originais
    n_jobs=-1
)

# 5) Treinar
stack.fit(X_train_filtered, y_train)

# 6) Avaliar
y_pred_train_log = stack.predict(X_train_filtered)
y_pred_val_log   = stack.predict(X_val_filtered)

y_pred_train = np.expm1(y_pred_train_log)
y_pred_val = np.expm1(y_pred_val_log)

mae_train = mean_absolute_error(y_train_orig, y_pred_train)
mae_val   = mean_absolute_error(y_val_orig, y_pred_val)

print("MAE train:", mae_train)
print("MAE val  :", mae_val)
print("gap :", mae_val-mae_train)

MAE train: 973.4257224284949
MAE val  : 1185.9134869036443
gap : 212.48776447514945


The results obtained are significantly better when compared to every other model alone. The MAE decreased significantly but so the overfitting, when comparing to our previous best model. Therefore, this is the model we're choosing for predicting the car prices on the test dataset.

# 3. Final Predictions

In [ ]:
test_final = test_filtered.copy()

In [ ]:
make_submission(
    model = stack,
    test_df = test_final,
    feature_cols = cols_selected,
    id_col="carID",
    output_path="../test_predictions/Group36_Version_Stack.csv",
    inverse_y_func=inverse_y_log1p,
    round_price=True,
)

,carID,price
0,89856,8363
1,106581,23817
2,80886,13906
3,100174,17361
4,81376,22276
...,...,...
32562,105775,17876
32563,81363,35568
32564,76833,35224
32565,91768,20909
